# Wastewater MAG / Metagenomic Datasets — HPC Downloader

This notebook fetches and downloads the datasets underlying six sources into a
`datasets/` tree, **one folder per paper/source**. It is designed to run for
**hours on an HPC Jupyter kernel** and to survive your laptop sleeping — the
kernel keeps running on the cluster.

**Sources and how each is mined (they are all different):**

| # | Source | Access method | Folder |
|---|--------|---------------|--------|
| 01 | Figshare 25016147 (Becsei/Martiny 5-city sewage) | Figshare REST API → all files + md5 | `01_figshare_5cities_becsei/` |
| 02 | OUP *Database* baaf089 → **ENA PRJEB68319** | ENA Portal API → assembly `contig.fa.gz` + metadata | `02_ena_PRJEB68319_assemblies/` |
| 03 | mSystems 00031-26 (Urban sewage resistomes) | Supplemental `.docx` + **Zenodo 14652833** | `03_msystems_resistome/` |
| 04 | NCBI taxon 527639 "wastewater metagenome", 2024–2026 | NCBI Datasets v2 API (CLI if available) → all ~5,389 genome FASTA | `04_ncbi_wastewater_metagenome_527639/` |
| 06 | PubMed 39215001 (Time-series sewage metagenomics) | **Same project as PRJEB68319** — manifest + link to 01/02 | `06_pubmed39215001_timeseries/` |
| 05 | Slack (bicbioeng) | Internal workspace — **manual**, placeholder note | `05_slack_manual/` |

**Design notes**
- Every download is **resumable** (HTTP Range) and **idempotent**: rerunning skips
  files already complete (size / md5 verified). Safe to re-run after a crash.
- md5 checks where the source publishes them (Figshare, ENA assemblies).
- Progress + errors are logged to `_logs/` and per-source `manifest_*.csv` files
  are written to `_manifests/`.
- Per your choices: ENA = **assemblies + metadata** (raw FASTQ links saved as a
  manifest but NOT downloaded); NCBI = **full genome FASTA for all ~5,389**.

**Before running:** see the "Environment check" cell — compute nodes sometimes
have no outbound internet. If the connectivity check fails, run this notebook on
a login / data-transfer node instead.


In [1]:
# =============================== CONFIG ===============================
# Edit BASE_DIR if you are NOT launching Jupyter from inside your `datasets/` dir.
import os
from pathlib import Path

# Default: current working directory (you said you are in `datasets/`).
BASE_DIR = Path(os.environ.get("MAG_BASE_DIR", Path.cwd()))
# If cwd is not literally named 'datasets', nest a datasets/ folder to be safe:
if BASE_DIR.name != "datasets":
    BASE_DIR = BASE_DIR / "datasets"

# Per-source folders
DIRS = {
    "figshare":  BASE_DIR / "01_figshare_5cities_becsei",
    "ena":       BASE_DIR / "02_ena_PRJEB68319_assemblies",
    "msystems":  BASE_DIR / "03_msystems_resistome",
    "ncbi":      BASE_DIR / "04_ncbi_wastewater_metagenome_527639",
    "slack":     BASE_DIR / "05_slack_manual",
    "pubmed":    BASE_DIR / "06_pubmed39215001_timeseries",
    "logs":      BASE_DIR / "_logs",
    "manifests": BASE_DIR / "_manifests",
}
for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

# Tunables
MAX_WORKERS   = int(os.environ.get("MAG_WORKERS", "6"))   # parallel downloads
HTTP_RETRIES  = 5
HTTP_TIMEOUT  = 120        # seconds per request
NCBI_YEARS    = (2024, 2026)   # inclusive release-year filter for taxon 527639
NCBI_TAXON    = 527639
DOWNLOAD_ENA_READS = False  # you chose assemblies + metadata only (True = also pull raw FASTQ, TBs!)

print("BASE_DIR =", BASE_DIR)
for k, v in DIRS.items():
    print(f"  {k:9s} -> {v}")


BASE_DIR = /mmfs2/home/usd.local/john.kangethe/datasets
  figshare  -> /mmfs2/home/usd.local/john.kangethe/datasets/01_figshare_5cities_becsei
  ena       -> /mmfs2/home/usd.local/john.kangethe/datasets/02_ena_PRJEB68319_assemblies
  msystems  -> /mmfs2/home/usd.local/john.kangethe/datasets/03_msystems_resistome
  ncbi      -> /mmfs2/home/usd.local/john.kangethe/datasets/04_ncbi_wastewater_metagenome_527639
  slack     -> /mmfs2/home/usd.local/john.kangethe/datasets/05_slack_manual
  pubmed    -> /mmfs2/home/usd.local/john.kangethe/datasets/06_pubmed39215001_timeseries
  logs      -> /mmfs2/home/usd.local/john.kangethe/datasets/_logs
  manifests -> /mmfs2/home/usd.local/john.kangethe/datasets/_manifests


In [2]:
# =============================== HELPERS ===============================
import sys, time, hashlib, logging, json, csv, urllib.request, urllib.error, io, zipfile
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

# ---- logging: file + stdout -----------------------------------------
LOG_FILE = DIRS["logs"] / f"download_{datetime.now():%Y%m%d_%H%M%S}.log"
logger = logging.getLogger("mags")
logger.setLevel(logging.INFO)
logger.handlers.clear()
_fmt = logging.Formatter("%(asctime)s %(levelname)s %(message)s", "%H:%M:%S")
for h in (logging.StreamHandler(sys.stdout), logging.FileHandler(LOG_FILE)):
    h.setFormatter(_fmt); logger.addHandler(h)
logger.info("Logging to %s", LOG_FILE)

USER_AGENT = "Cowork-MAG-downloader/1.0 (mailto:ngangajohn536@gmail.com)"

def _open(url, headers=None, timeout=HTTP_TIMEOUT):
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT, **(headers or {})})
    return urllib.request.urlopen(req, timeout=timeout)

def http_bytes(url, retries=HTTP_RETRIES):
    """GET url -> raw bytes, with retries."""
    for attempt in range(1, retries + 1):
        try:
            with _open(url) as r:
                return r.read()
        except Exception as e:
            logger.warning("GET fail (%d/%d) %s : %s", attempt, retries, url, e)
            time.sleep(min(2 ** attempt, 30))
    raise RuntimeError(f"GET failed after {retries}: {url}")

def http_json(url, retries=HTTP_RETRIES):
    return json.loads(http_bytes(url, retries).decode("utf-8", "replace"))

def http_text(url, retries=HTTP_RETRIES):
    return http_bytes(url, retries).decode("utf-8", "replace")

def md5_of(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def download_file(url, dest, expected_md5=None, expected_size=None, retries=HTTP_RETRIES):
    """Streaming, resumable, idempotent download.
    Returns 'skip' | 'ok' | 'fail'. Verifies md5/size when provided."""
    dest = Path(dest); dest.parent.mkdir(parents=True, exist_ok=True)
    # already complete?
    if dest.exists():
        if expected_md5 and md5_of(dest) == expected_md5:
            return "skip"
        if expected_size and dest.stat().st_size == int(expected_size) and not expected_md5:
            return "skip"
        if not expected_md5 and not expected_size and dest.stat().st_size > 0:
            return "skip"
    part = dest.with_suffix(dest.suffix + ".part")
    for attempt in range(1, retries + 1):
        try:
            have = part.stat().st_size if part.exists() else 0
            headers = {"Range": f"bytes={have}-"} if have else {}
            with _open(url, headers=headers) as r:
                mode = "ab" if (have and r.status == 206) else "wb"
                if mode == "wb":
                    have = 0
                with open(part, mode) as f:
                    while True:
                        chunk = r.read(1 << 20)
                        if not chunk:
                            break
                        f.write(chunk)
            # verify
            if expected_md5:
                if md5_of(part) != expected_md5:
                    logger.warning("md5 mismatch %s -> retry", dest.name)
                    part.unlink(missing_ok=True); raise ValueError("md5")
            elif expected_size and part.stat().st_size != int(expected_size):
                logger.warning("size mismatch %s (%d != %s) -> retry",
                               dest.name, part.stat().st_size, expected_size)
                part.unlink(missing_ok=True); raise ValueError("size")
            part.replace(dest)
            return "ok"
        except Exception as e:
            logger.warning("dl fail (%d/%d) %s : %s", attempt, retries, dest.name, e)
            time.sleep(min(2 ** attempt, 30))
    return "fail"

def write_manifest(name, rows, header):
    path = DIRS["manifests"] / name
    with open(path, "w", newline="") as f:
        w = csv.writer(f); w.writerow(header); w.writerows(rows)
    logger.info("manifest -> %s (%d rows)", path, len(rows))
    return path

def run_pool(items, fn, workers=MAX_WORKERS, label="tasks"):
    """Map fn over items with a thread pool; returns list of results. Logs progress."""
    results, done, total = [], 0, len(items)
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = {ex.submit(fn, it): it for it in items}
        for fut in as_completed(futs):
            done += 1
            try:
                results.append(fut.result())
            except Exception as e:
                results.append(("fail", str(e)))
            if done % 25 == 0 or done == total:
                logger.info("[%s] %d/%d", label, done, total)
    return results

def human(n):
    n = float(n)
    for u in "B KB MB GB TB PB".split():
        if n < 1024: return f"{n:.1f}{u}"
        n /= 1024
    return f"{n:.1f}EB"

print("Helpers ready. Log file:", LOG_FILE)


01:38:19 INFO Logging to /mmfs2/home/usd.local/john.kangethe/datasets/_logs/download_20260707_013819.log
Helpers ready. Log file: /mmfs2/home/usd.local/john.kangethe/datasets/_logs/download_20260707_013819.log


In [4]:
# ===================== ENVIRONMENT / CONNECTIVITY CHECK =====================
# Compute nodes often lack outbound internet. Verify before the long run.
CHECKS = {
    "Figshare API": "https://api.figshare.com/v2/articles/25016147",
    "ENA Portal":   "https://www.ebi.ac.uk/ena/portal/api/",
    "NCBI Datasets":"https://api.ncbi.nlm.nih.gov/datasets/v2/version",
    "Zenodo":       "https://zenodo.org/api/records/14652833",
    "ASM journals": "https://journals.asm.org/",
}
ok_all = True
for name, url in CHECKS.items():
    try:
        with _open(url, timeout=30) as r:
            print(f"  OK   {name:15s} HTTP {r.status}")
    except Exception as e:
        ok_all = False
        print(f"  FAIL {name:15s} {e}")
if not ok_all:
    print("\n*** Some endpoints are unreachable. If ALL fail, this node likely has no")
    print("*** outbound internet. Run this notebook on a login/data-transfer node,")
    print("*** or load an http_proxy module, then re-run. Downloads are resumable.")
else:
    print("\nAll endpoints reachable — good to go.")


  OK   Figshare API    HTTP 200
  OK   ENA Portal      HTTP 200
  OK   NCBI Datasets   HTTP 200
  OK   Zenodo          HTTP 200
  FAIL ASM journals    HTTP Error 403: Forbidden

*** Some endpoints are unreachable. If ALL fail, this node likely has no
*** outbound internet. Run this notebook on a login/data-transfer node,
*** or load an http_proxy module, then re-run. Downloads are resumable.


## Run

Run the source cells below top-to-bottom. Each is independent and resumable; if one fails, fix and re-run just that cell.

In [5]:
# ============ SOURCE 01 — FIGSHARE 25016147 (5-city sewage, Becsei) ============
# Figshare REST API lists every file with a published md5 -> download + verify.
FIGSHARE_ARTICLE = 25016147
meta = http_json(f"https://api.figshare.com/v2/articles/{FIGSHARE_ARTICLE}")
files = meta["files"]
logger.info("Figshare article %s: %d files, %s total",
            FIGSHARE_ARTICLE, len(files), human(meta.get("size", 0)))

# Save the article metadata (rich provenance) alongside the data
(DIRS["figshare"] / "figshare_article_metadata.json").write_text(json.dumps(meta, indent=2))

def _get_figshare(fobj):
    dest = DIRS["figshare"] / fobj["name"]
    status = download_file(fobj["download_url"], dest,
                           expected_md5=fobj.get("supplied_md5"),
                           expected_size=fobj.get("size"))
    logger.info("[figshare] %-8s %s (%s)", status, fobj["name"], human(fobj.get("size", 0)))
    return (fobj["name"], fobj.get("size"), fobj.get("supplied_md5"), status)

rows = run_pool(files, _get_figshare, label="figshare")
write_manifest("manifest_01_figshare.csv", rows, ["file", "size_bytes", "md5", "status"])
print("Figshare done:", {s: [r[3] for r in rows].count(s) for s in ("ok","skip","fail")})




!python build_tree_view.py .


01:39:29 INFO Figshare article 25016147: 20 files, 1.7GB total
01:39:31 INFO [figshare] ok       genomic_phylum_abundance.csv (69.0KB)
01:39:31 INFO [figshare] ok       genomic_class_abundance.csv (152.9KB)
01:39:31 INFO [figshare] ok       meta_location.csv (30.0KB)
01:39:31 INFO [figshare] ok       genomic_order_abundance.csv (332.3KB)
01:39:31 INFO [figshare] ok       genomic_family_abundance.csv (705.8KB)
01:39:31 INFO [figshare] ok       genomic_genus_abundance.csv (2.3MB)
01:39:32 INFO [figshare] ok       qpcr.csv (16.9KB)
01:39:32 INFO [figshare] ok       resfinder_class_abundance.csv (26.9KB)
01:39:33 INFO [figshare] ok       mag_checkm2.csv (2.4MB)
01:39:33 INFO [figshare] ok       proximeta.csv (373.1KB)
01:39:33 INFO [figshare] ok       genomic_species_abundance.csv (10.0MB)
01:39:33 INFO [figshare] ok       resfinder_gene_abundance.csv (594.5KB)
01:39:33 INFO [figshare] ok       gene_annotation.csv (24.8MB)
01:39:34 INFO [figshare] ok       resfinder_gene_class.csv (285.6KB

In [ ]:
# ============ SOURCE 02 — ENA PRJEB68319 (assemblies = MAG contigs) ============
# ENA Portal API. We download per-sample assembly contigs (contig.fa.gz) and
# save full run/sample metadata. Raw FASTQ links are saved as a manifest only
# (DOWNLOAD_ENA_READS = False).
ENA_PROJECT = "PRJEB68319"
ENA_API = "https://www.ebi.ac.uk/ena/portal/api/filereport"

def ena_report(result, fields):
    url = (f"{ENA_API}?accession={ENA_PROJECT}&result={result}"
           f"&fields={','.join(fields)}&format=tsv&limit=0")
    text = http_text(url)
    lines = [l for l in text.splitlines() if l.strip()]
    hdr = lines[0].split("\t")
    return [dict(zip(hdr, l.split("\t"))) for l in lines[1:]]

# --- 2a. Assemblies (analysis) ---
an_fields = ["analysis_accession","analysis_type","sample_accession",
             "analysis_title","generated_ftp","generated_md5","generated_bytes"]
analyses = ena_report("analysis", an_fields)
logger.info("ENA %s: %d analysis records", ENA_PROJECT, len(analyses))
write_manifest("manifest_02_ena_assemblies.csv",
               [[a.get(k,"") for k in an_fields] for a in analyses], an_fields)

def _get_ena(a):
    ftp = a.get("generated_ftp","")
    if not ftp:
        return (a["analysis_accession"], "", "nolink")
    # a record may list several files separated by ';'
    statuses = []
    ftps = ftp.split(";")
    md5s = (a.get("generated_md5","") or "").split(";")
    for i, fp in enumerate(ftps):
        if not fp: continue
        url = "https://" + fp
        fname = f"{a['analysis_accession']}__{Path(fp).name}"
        md5 = md5s[i] if i < len(md5s) and md5s[i] else None
        st = download_file(url, DIRS["ena"] / "assemblies" / fname, expected_md5=md5)
        statuses.append(st)
    return (a["analysis_accession"], ftp, ",".join(statuses))

ares = run_pool(analyses, _get_ena, label="ena-asm")
_st = [s for _,_,st in ares for s in st.split(",") if st not in ("nolink","")]
print("ENA assemblies:", {s: _st.count(s) for s in ("ok","skip","fail")}, "| records:", len(ares))

# --- 2b. Raw-read metadata (manifest only; no download unless DOWNLOAD_ENA_READS) ---
rr_fields = ["run_accession","sample_accession","fastq_ftp","fastq_bytes",
             "fastq_md5","library_strategy","instrument_platform","read_count"]
runs = ena_report("read_run", rr_fields)
write_manifest("manifest_02_ena_reads.csv",
               [[r.get(k,"") for k in rr_fields] for r in runs], rr_fields)
logger.info("ENA read_run: %d runs catalogued (FASTQ NOT downloaded)", len(runs))

if DOWNLOAD_ENA_READS:
    logger.warning("DOWNLOAD_ENA_READS=True -> pulling raw FASTQ (this can be TBs!)")
    def _get_reads(r):
        out = []
        ftps = (r.get("fastq_ftp","") or "").split(";")
        md5s = (r.get("fastq_md5","") or "").split(";")
        for i, fp in enumerate(ftps):
            if not fp: continue
            md5 = md5s[i] if i < len(md5s) else None
            out.append(download_file("https://"+fp, DIRS["ena"]/"reads"/Path(fp).name, expected_md5=md5))
        return (r["run_accession"], ",".join(out))
    run_pool(runs, _get_reads, label="ena-reads")

print("ENA done. Assemblies in:", DIRS["ena"] / "assemblies")


01:40:54 INFO ENA PRJEB68319: 38128 analysis records
01:40:54 INFO manifest -> /mmfs2/home/usd.local/john.kangethe/datasets/_manifests/manifest_02_ena_assemblies.csv (38128 rows)


In [ ]:
# ======= SOURCE 03 — mSystems 00031-26 (Urban sewage resistomes) =======
# Two artifacts: (a) supplemental .docx (Table S4 lists ENA accessions),
# (b) Zenodo record 14652833 (mapstat / read-based ARG results).
DOI = "10.1128/msystems.00031-26"

# --- 3a. Supplemental files (ASM) ---
suppl = {
    "msystems.00031-26-s0001.docx":
        f"https://journals.asm.org/doi/suppl/{DOI}/suppl_file/msystems.00031-26-s0001.docx",
    "msystems.00031-26-s0002.docx":
        f"https://journals.asm.org/doi/suppl/{DOI}/suppl_file/msystems.00031-26-s0002.docx",
}
srows = []
for name, url in suppl.items():
    st = download_file(url, DIRS["msystems"] / name)
    logger.info("[msystems suppl] %-6s %s", st, name)
    srows.append([name, url, st])

# --- 3b. Zenodo record 14652833 (all files via Zenodo API) ---
try:
    zrec = http_json("https://zenodo.org/api/records/14652833")
    (DIRS["msystems"] / "zenodo_14652833_metadata.json").write_text(json.dumps(zrec, indent=2))
    zfiles = zrec.get("files", [])
    logger.info("Zenodo 14652833: %d files", len(zfiles))
    for zf in zfiles:
        url = zf.get("links", {}).get("self") or zf.get("links", {}).get("download")
        name = zf.get("key") or zf.get("filename")
        md5 = (zf.get("checksum","") or "").replace("md5:", "") or None
        st = download_file(url, DIRS["msystems"] / "zenodo_14652833" / name,
                           expected_md5=md5, expected_size=zf.get("size"))
        logger.info("[zenodo] %-6s %s (%s)", st, name, human(zf.get("size", 0)))
        srows.append([name, url, st])
except Exception as e:
    logger.warning("Zenodo fetch problem: %s (check record 14652833 manually)", e)

write_manifest("manifest_03_msystems.csv", srows, ["file", "url", "status"])
print("mSystems done. Table S4 (ENA accessions) is inside msystems.00031-26-s0002.docx")



!python build_tree_view.py .


In [ ]:
# ==== SOURCE 04 — NCBI taxon 527639 "wastewater metagenome", 2024-2026 ====
# All ~5,389 genome assemblies. Prefer the official `datasets` CLI (fast, bulk,
# dehydrated+rehydrate). Fall back to pure-Python REST (per-accession, resumable).
import shutil, subprocess

DATASETS_V2 = "https://api.ncbi.nlm.nih.gov/datasets/v2"
ncbi_dir = DIRS["ncbi"]

# ---- Step 1: collect ALL accessions + metadata, filter by release year ----
def collect_reports():
    reports, token, page = [], None, 0
    while True:
        url = (f"{DATASETS_V2}/genome/taxon/{NCBI_TAXON}/dataset_report"
               f"?page_size=1000&filters.assembly_source=all")
        if token:
            url += f"&page_token={token}"
        js = http_json(url)
        reports += js.get("reports", [])
        page += 1
        token = js.get("next_page_token")
        logger.info("NCBI report page %d: cumulative %d / %s",
                    page, len(reports), js.get("total_count","?"))
        if not token:
            break
    return reports

logger.info("Collecting NCBI dataset report for taxon %s ...", NCBI_TAXON)
reports = collect_reports()

def _year(rep):
    d = rep.get("assembly_info", {}).get("release_date", "")
    try: return int(d[:4])
    except: return None

lo, hi = NCBI_YEARS
kept = [r for r in reports if (_year(r) is not None and lo <= _year(r) <= hi)]
accessions = [r["accession"] for r in kept]
logger.info("NCBI: %d total assemblies, %d within %d-%d", len(reports), len(kept), lo, hi)

# metadata manifest + full JSONL
meta_rows = [[r.get("accession",""),
              r.get("organism",{}).get("organism_name",""),
              r.get("assembly_info",{}).get("release_date",""),
              r.get("assembly_info",{}).get("assembly_level",""),
              r.get("assembly_info",{}).get("bioproject_accession",""),
              r.get("assembly_stats",{}).get("total_sequence_length",""),
              r.get("source_database","")] for r in kept]
write_manifest("manifest_04_ncbi.csv", meta_rows,
    ["accession","organism","release_date","assembly_level","bioproject","total_len","source_db"])
with open(ncbi_dir / "ncbi_527639_reports.jsonl", "w") as f:
    for r in kept:
        f.write(json.dumps(r) + "\n")

# ---- Step 2: download genome FASTA ----
seq_dir = ncbi_dir / "genomes"; seq_dir.mkdir(parents=True, exist_ok=True)
datasets_bin = shutil.which("datasets")

if datasets_bin:
    logger.info("Using NCBI `datasets` CLI at %s (dehydrated bulk download)", datasets_bin)
    zip_path = ncbi_dir / "ncbi_527639.zip"
    cmd = [datasets_bin, "download", "genome", "taxon", str(NCBI_TAXON),
           "--released-after", f"{lo-1}-12-31", "--released-before", f"{hi}-12-31",
           "--include", "genome", "--dehydrated", "--filename", str(zip_path)]
    logger.info("CLI: %s", " ".join(cmd))
    subprocess.run(cmd, check=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(ncbi_dir / "ncbi_dataset_pkg")
    subprocess.run([datasets_bin, "rehydrate", "--directory",
                    str(ncbi_dir / "ncbi_dataset_pkg"), "--max-workers", str(MAX_WORKERS)],
                   check=True)
    logger.info("CLI rehydrate complete -> %s", ncbi_dir / "ncbi_dataset_pkg")
else:
    logger.info("`datasets` CLI not found -> pure-Python REST, per-accession (resumable).")
    def _get_genome(acc):
        url = (f"{DATASETS_V2}/genome/accession/{acc}/download"
               f"?include_annotation_type=GENOME_FASTA")
        st = download_file(url, seq_dir / f"{acc}.zip")  # each zip = one genome FASTA
        return (acc, st)
    gres = run_pool(accessions, _get_genome, workers=MAX_WORKERS, label="ncbi-dl")
    counts = {s: [st for _,st in gres].count(s) for s in ("ok","skip","fail")}
    logger.info("NCBI genome downloads: %s", counts)
    print("NCBI done:", counts, "-> zips in", seq_dir)
    print("   (each .zip holds ncbi_dataset/data/<acc>/*.fna ; unzip when needed)")

print("NCBI step complete. Metadata: manifest_04_ncbi.csv")


In [ ]:
# === SOURCE 06 — PubMed 39215001 (Time-series sewage metagenomics) ===
# This is the RESEARCH paper for the SAME 5-city project as sources 01 & 02.
# Its MAGs / assemblies = ENA PRJEB68319 (already fetched in folder 02) and its
# supplementary tables = the Figshare record (folder 01). To avoid duplicating
# tens of GB we write a manifest + README pointing at 01/02 rather than re-downloading.
readme = f"""PubMed 39215001 — "Time-series sewage metagenomics distinguishes seasonal,
human-derived and environmental microbial communities..." (Martiny et al.)

This paper describes the SAME longitudinal 5-city European sewage project as:
  - Source 01 (Figshare 25016147)  -> ../01_figshare_5cities_becsei/
  - Source 02 (ENA PRJEB68319)     -> ../02_ena_PRJEB68319_assemblies/

MAGs / metagenome assemblies for this paper == ENA PRJEB68319 (2,332 MAGs; the
per-sample assemblies are downloaded in folder 02). Derived MAG tables
(mag_checkm2.csv, binned_contig.csv, gene_annotation.csv) are in folder 01.

No separate download is required. Set REDOWNLOAD_PUBMED=True in this cell to pull
a second independent copy of the PRJEB68319 assemblies into this folder.
"""
(DIRS["pubmed"] / "README.txt").write_text(readme)

REDOWNLOAD_PUBMED = False
if REDOWNLOAD_PUBMED:
    logger.info("Re-downloading PRJEB68319 assemblies into folder 06 ...")
    for a in analyses:  # reuses `analyses` from the ENA cell (run that cell first)
        ftp = a.get("generated_ftp","")
        for fp in ftp.split(";"):
            if fp:
                download_file("https://"+fp, DIRS["pubmed"]/"assemblies"/f"{a['analysis_accession']}__{Path(fp).name}")
else:
    # leave a symlink to the shared ENA assemblies for convenience
    link = DIRS["pubmed"] / "ena_assemblies_link"
    try:
        if not link.exists():
            link.symlink_to(DIRS["ena"] / "assemblies", target_is_directory=True)
    except Exception as e:
        logger.info("symlink skipped (%s)", e)

print("PubMed 39215001 mapped to ENA PRJEB68319 (folder 02) + Figshare (folder 01). See README.txt")



!python build_tree_view.py .


In [20]:
# ===== SOURCE 07 — Nature s41467-024-51957-8 (MAGs, ENA PRJEB68319, ALL metadata) =====
# Published version of PubMed 39215001 / same 5-city time-series study.
# MAGs deposited under PRJEB68319. Pull EVERY analysis file (MAGs/assemblies) into a
# dedicated folder + COMPLETE ENA metadata (all fields, all result types).
import urllib.error, time
NATURE_DIR = BASE_DIR / "07_nature_s41467_MAGs"
(NATURE_DIR / "assemblies").mkdir(parents=True, exist_ok=True)
(NATURE_DIR / "metadata").mkdir(parents=True, exist_ok=True)
ENA_PROJECT = "PRJEB68319"
ENA_API = "https://www.ebi.ac.uk/ena/portal/api"

def ena_all_fields(result):
    """Every available ENA metadata field name for a result type (or None)."""
    try:
        js = http_json(f"{ENA_API}/returnFields?result={result}&format=json")
        keys = [(d.get("columnId") or d.get("id")) for d in js]
        return [k for k in keys if k]
    except Exception as e:
        logger.warning("returnFields(%s) failed (%s) -> ENA default field set", result, e)
        return None

# ===== FIX 1: 429-aware field probe (REPLACE _fields_ok in cell 7) =====

def _fields_ok(result, fields):
    """True if filereport accepts these fields. Only a real HTTP 400 means
    'invalid field(s)'; 429/5xx/timeouts are transient -> back off & retry, so a
    rate-limit can never masquerade as an invalid field (that bug nuked ALL sample fields)."""
    url = (f"{ENA_API}/filereport?accession={ENA_PROJECT}&result={result}"
           f"&fields={','.join(fields)}&format=tsv&limit=0")
    for attempt in range(6):
        try:
            with _open(url, timeout=60) as r:
                return r.status == 200
        except urllib.error.HTTPError as e:
            if e.code == 400:
                return False
            time.sleep(min(2 ** attempt, 30))
        except Exception:
            time.sleep(min(2 ** attempt, 30))
    return False

def valid_fields(result, fields):
    """Largest subset of `fields` that filereport accepts (chunk + bisect)."""
    good = []
    def check(batch):
        if not batch:
            return
        if _fields_ok(result, batch):
            good.extend(batch)
        elif len(batch) == 1:
            logger.info("  drop invalid field for %s: %s", result, batch[0])
        else:
            mid = len(batch) // 2
            check(batch[:mid]); check(batch[mid:])
    for i in range(0, len(fields), 25):     # probe in chunks to limit requests
        check(fields[i:i + 25])
    return good

def ena_dump_metadata(result):
    fields = ena_all_fields(result) or []
    if fields and not _fields_ok(result, fields):     # only prune if the full set fails
        logger.info("[nature meta] %s: full field set rejected, pruning...", result)
        fields = valid_fields(result, fields)
    fld = ("&fields=" + ",".join(fields)) if fields else ""
    url = f"{ENA_API}/filereport?accession={ENA_PROJECT}&result={result}{fld}&format=tsv&limit=0"
    txt = http_text(url)
    out = NATURE_DIR / "metadata" / f"{ENA_PROJECT}_{result}_ALLfields.tsv"
    out.write_text(txt)
    n = max(0, len([l for l in txt.splitlines() if l.strip()]) - 1)
    logger.info("[nature meta] %-15s %5d rows, %3d fields -> %s",
                result, n, len(fields), out.name)

# 1) COMPLETE metadata for every result type tied to the project
for result in ("analysis", "read_run", "read_experiment", "sample", "study"):
    try:
        ena_dump_metadata(result)
    except Exception as e:
        logger.warning("metadata %s failed: %s", result, e)
# full project/study XML (richest provenance)
try:
    xml_txt = http_text(f"https://www.ebi.ac.uk/ena/browser/api/xml/{ENA_PROJECT}")
    (NATURE_DIR / "metadata" / f"{ENA_PROJECT}_study.xml").write_text(xml_txt)
except Exception as e:
    logger.warning("study XML failed: %s", e)

# 2) Download ALL analysis files (the MAGs/assemblies); reuse folder 02 via hardlink.
an_fields = ["analysis_accession","analysis_type","assembly_type","sample_accession",
             "analysis_title","generated_ftp","generated_md5","generated_bytes"]
txt = http_text(f"{ENA_API}/filereport?accession={ENA_PROJECT}&result=analysis"
                f"&fields={','.join(an_fields)}&format=tsv&limit=0")
lines = [l for l in txt.splitlines() if l.strip()]; hdr = lines[0].split("\t")
records = [dict(zip(hdr, l.split("\t"))) for l in lines[1:]]
logger.info("Nature/ENA: %d analysis records", len(records))
write_manifest("manifest_07_nature_ena_analysis.csv",
               [[r.get(k,"") for k in an_fields] for r in records], an_fields)

src02 = BASE_DIR / "02_ena_PRJEB68319_assemblies" / "assemblies"
def _fetch_nature(r):
    outs = []
    ftps = (r.get("generated_ftp","") or "").split(";")
    md5s = (r.get("generated_md5","") or "").split(";")
    for i, fp in enumerate(ftps):
        if not fp:
            continue
        fname = f"{r['analysis_accession']}__{Path(fp).name}"
        dest = NATURE_DIR / "assemblies" / fname
        if dest.exists():
            outs.append("skip"); continue
        reuse = src02 / fname
        if reuse.exists():
            try:
                os.link(reuse, dest); outs.append("link"); continue
            except Exception:
                try:
                    dest.symlink_to(reuse); outs.append("symlink"); continue
                except Exception:
                    pass
        md5 = md5s[i] if i < len(md5s) and md5s[i] else None
        outs.append(download_file("https://" + fp, dest, expected_md5=md5))
    return (r["analysis_accession"], ",".join(outs))

res = run_pool(records, _fetch_nature, label="nature-mags")
flat = [s for _, st in res for s in st.split(",") if s]
print("Nature MAGs/assemblies:", {k: flat.count(k) for k in ("ok","skip","link","symlink","fail")})
print("Files ->", NATURE_DIR / "assemblies", "| metadata ->", NATURE_DIR / "metadata")



!python build_tree_view.py .

12:27:38 INFO [nature meta] analysis        38128 rows, 137 fields -> PRJEB68319_analysis_ALLfields.tsv
12:27:40 INFO [nature meta] read_run          278 rows, 195 fields -> PRJEB68319_read_run_ALLfields.tsv
12:27:42 INFO [nature meta] read_experiment   278 rows, 195 fields -> PRJEB68319_read_experiment_ALLfields.tsv
12:27:43 INFO [nature meta] sample: full field set rejected, pruning...
12:27:45 INFO   drop invalid field for sample: age
12:27:46 INFO   drop invalid field for sample: altitude
12:27:47 INFO   drop invalid field for sample: assembly_quality
12:27:47 INFO   drop invalid field for sample: assembly_software
12:27:48 INFO   drop invalid field for sample: binning_software
12:27:49 INFO   drop invalid field for sample: bio_material
12:27:50 INFO   drop invalid field for sample: broad_scale_environmental_context
12:27:51 INFO   drop invalid field for sample: broker_name
12:27:52 INFO   drop invalid field for sample: cell_line
12:27:53 INFO   drop invalid field for sample: cell_

In [21]:
# ===== FIX 2 for Cell 7: complete SAMPLE metadata via per-sample XML (NEW cell after cell 7) =====
import re
NATURE_DIR = BASE_DIR / "07_nature_s41467_MAGs"
sx_dir = NATURE_DIR / "metadata" / "sample_xml"; sx_dir.mkdir(parents=True, exist_ok=True)
samp = set()
for name in ("PRJEB68319_read_run_ALLfields.tsv", "PRJEB68319_analysis_ALLfields.tsv"):
    p = NATURE_DIR / "metadata" / name
    if p.exists():
        head, *rows = p.read_text().splitlines()
        cols = head.split("\t")
        if "sample_accession" in cols:
            j = cols.index("sample_accession")
            for r in rows:
                cells = r.split("\t")
                v = cells[j] if j < len(cells) else ""
                for a in re.split(r"[;,\s]+", v):
                    if a.startswith("SAM"): samp.add(a)
samp = sorted(samp)
logger.info("Fetching XML for %d unique samples", len(samp))
def _get_sample_xml(acc):
    dest = sx_dir / f"{acc}.xml"
    if dest.exists() and dest.stat().st_size > 0: return (acc, "skip")
    try:
        dest.write_text(http_text(f"https://www.ebi.ac.uk/ena/browser/api/xml/{acc}"))
        return (acc, "ok")
    except Exception as e:
        return (acc, f"fail:{e}")
run_pool(samp, _get_sample_xml, workers=4, label="sample-xml")
print("Sample XML ->", sx_dir, "| files:", len(list(sx_dir.glob('*.xml'))))





!python build_tree_view.py .

12:31:35 INFO Fetching XML for 267 unique samples
12:31:38 INFO [sample-xml] 25/267
12:31:41 INFO [sample-xml] 50/267
12:31:44 INFO [sample-xml] 75/267
12:31:47 INFO [sample-xml] 100/267
12:31:50 INFO [sample-xml] 125/267
12:31:53 INFO [sample-xml] 150/267
12:31:56 INFO [sample-xml] 175/267
12:31:59 INFO [sample-xml] 200/267
12:32:02 INFO [sample-xml] 225/267
12:32:05 INFO [sample-xml] 250/267
12:32:07 INFO [sample-xml] 267/267
Sample XML -> /mmfs2/home/usd.local/john.kangethe/datasets/07_nature_s41467_MAGs/metadata/sample_xml | files: 267


In [23]:
# ===== ONE-TIME REPAIR CELL: Fixes 3, 4, 5 =====
import csv, gzip, time

# --- FIX 3: recover the 2 ENA assemblies whose published md5 was stale ---
bad_erz = ["ERZ22682287", "ERZ22682989"]     # add any other 'fail : md5' from logs
ENA_ASM = BASE_DIR / "02_ena_PRJEB68319_assemblies" / "assemblies"
with open(BASE_DIR / "_manifests" / "manifest_02_ena_assemblies.csv") as f:
    man = {r["analysis_accession"]: r for r in csv.DictReader(f)}
for erz in bad_erz:
    r = man.get(erz)
    if not r:
        logger.warning("%s not in manifest_02", erz); continue
    for fp in (r.get("generated_ftp", "") or "").split(";"):
        if not fp: continue
        dest = ENA_ASM / f"{erz}__{Path(fp).name}"
        st = download_file("https://" + fp, dest)          # size-verify only (no md5)
        try:
            with gzip.open(dest) as g: g.read(1 << 20)      # validate gzip integrity
            logger.info("recovered %s (%s, gzip OK)", dest.name, st)
        except Exception as ex:
            logger.warning("still bad after refetch: %s (%s)", dest.name, ex)

# --- FIX 4: fetch the NCBI genomes that hit 429 (sequential + optional API key) ---
missing_ncbi = ["GCA_963851715.1", "GCA_963957095.1", "GCA_963962975.1",
                "GCA_977093805.1", "GCA_977096405.1"]
seq_dir = BASE_DIR / "04_ncbi_wastewater_metagenome_527639" / "genomes"
NCBI_KEY = os.environ.get("NCBI_API_KEY")     # optional: set to raise the rate limit
for acc in missing_ncbi:
    url = (f"https://api.ncbi.nlm.nih.gov/datasets/v2/genome/accession/{acc}"
           f"/download?include_annotation_type=GENOME_FASTA")
    if NCBI_KEY: url += f"&api_key={NCBI_KEY}"
    print(acc, download_file(url, seq_dir / f"{acc}.zip"))
    time.sleep(1)                              # be gentle to avoid 429

# --- FIX 5: mSystems supplemental .docx (ASM 403s scripts) ---
msys = BASE_DIR / "03_msystems_resistome"
DOI = "10.1128/msystems.00031-26"
suppl = {
    "msystems.00031-26-s0001.docx": f"https://journals.asm.org/doi/suppl/{DOI}/suppl_file/msystems.00031-26-s0001.docx",
    "msystems.00031-26-s0002.docx": f"https://journals.asm.org/doi/suppl/{DOI}/suppl_file/msystems.00031-26-s0002.docx",
}
hdr = {"Referer": f"https://journals.asm.org/doi/full/{DOI}",
       "Accept": "application/msword,application/octet-stream,*/*",
       "Accept-Language": "en-US,en;q=0.9"}
for name, url in suppl.items():
    dest = msys / name
    if dest.exists() and dest.stat().st_size > 0:
        print("skip", name); continue
    try:
        with _open(url, headers=hdr, timeout=90) as r:
            dest.write_bytes(r.read())
        print("ok", name, human(dest.stat().st_size))
    except Exception as e:
        print("BLOCKED -> download manually in a browser, drop into 03/:", name, "|", e)
        print("   ", url)


!python build_tree_view.py .

12:41:26 WARNING still bad after refetch: ERZ22682287__contig.fa.gz (Not a gzipped file (b'<!'))
12:41:26 WARNING still bad after refetch: ERZ22682989__contig.fa.gz (Not a gzipped file (b'<!'))
GCA_963851715.1 skip
GCA_963957095.1 skip
GCA_963962975.1 skip
GCA_977093805.1 skip
GCA_977096405.1 skip
BLOCKED -> download manually in a browser, drop into 03/: msystems.00031-26-s0001.docx | HTTP Error 403: Forbidden
    https://journals.asm.org/doi/suppl/10.1128/msystems.00031-26/suppl_file/msystems.00031-26-s0001.docx
BLOCKED -> download manually in a browser, drop into 03/: msystems.00031-26-s0002.docx | HTTP Error 403: Forbidden
    https://journals.asm.org/doi/suppl/10.1128/msystems.00031-26/suppl_file/msystems.00031-26-s0002.docx


In [24]:
# ===== SOURCE 08 — bioRxiv 2025.07.21.666059v1 (reanalysis, NO new data) =====
BIORXIV_DIR = BASE_DIR / "08_biorxiv_666059_reanalysis"
BIORXIV_DIR.mkdir(parents=True, exist_ok=True)
readme = """bioRxiv 10.1101/2025.07.21.666059v1 (Q-net digital twin of sewage microbial dynamics)

DATA AVAILABILITY: generates NO new sequencing data. Section 4.1 states it is a
computational REANALYSIS of the previously published longitudinal 5-city sewage
dataset (7 treatment plants; 2,332 MAGs incl. 1,334 novel species) -- i.e. the
SAME data as:
  - ENA PRJEB68319          -> ../07_nature_s41467_MAGs/  (and ../02_ena_PRJEB68319_assemblies/)
  - Figshare 25016147       -> ../01_figshare_5cities_becsei/
  - Nature s41467-024-51957-8 / PubMed 39215001

No separate dataset exists to download. The abundance tables it models are the
Figshare genomic_*_abundance.csv files (folder 01).
"""
(BIORXIV_DIR / "README_no_new_data.txt").write_text(readme)
try:  # keep a copy of the preprint full text for the record
    html = http_text("https://www.biorxiv.org/content/10.1101/2025.07.21.666059v1.full")
    (BIORXIV_DIR / "biorxiv_666059_fulltext.html").write_text(html)
except Exception as e:
    logger.warning("biorxiv fetch skipped: %s", e)
print(readme)


!python build_tree_view.py .

12:41:39 WARNING GET fail (1/5) https://www.biorxiv.org/content/10.1101/2025.07.21.666059v1.full : HTTP Error 403: Forbidden
12:41:41 WARNING GET fail (2/5) https://www.biorxiv.org/content/10.1101/2025.07.21.666059v1.full : HTTP Error 403: Forbidden
12:41:45 WARNING GET fail (3/5) https://www.biorxiv.org/content/10.1101/2025.07.21.666059v1.full : HTTP Error 403: Forbidden
12:41:53 WARNING GET fail (4/5) https://www.biorxiv.org/content/10.1101/2025.07.21.666059v1.full : HTTP Error 403: Forbidden
12:42:09 WARNING GET fail (5/5) https://www.biorxiv.org/content/10.1101/2025.07.21.666059v1.full : HTTP Error 403: Forbidden
12:42:39 WARNING biorxiv fetch skipped: GET failed after 5: https://www.biorxiv.org/content/10.1101/2025.07.21.666059v1.full
bioRxiv 10.1101/2025.07.21.666059v1 (Q-net digital twin of sewage microbial dynamics)

DATA AVAILABILITY: generates NO new sequencing data. Section 4.1 states it is a
computational REANALYSIS of the previously published longitudinal 5-city sewage
dat

In [25]:
# =============================== SUMMARY ===============================
# Walk EVERY source folder under BASE_DIR (auto-includes 07, 08, and anything you
# add manually). Dedupe hardlinks/symlinks by inode so TOTAL reflects real disk use.
SKIP = {"_logs", "_manifests"}
master, seen = [], set()
grand_files = grand_listed = grand_unique = 0
print(f"{'source':40s} {'files':>7s} {'size':>12s}")
print("-" * 62)
for d in sorted(p for p in BASE_DIR.iterdir() if p.is_dir() and p.name not in SKIP):
    files = [p for p in d.rglob("*") if p.is_file()]
    nbytes = 0
    for p in files:
        try:
            stt = p.stat()
        except OSError:
            continue
        nbytes += stt.st_size
        key = (stt.st_dev, stt.st_ino)
        if key not in seen:                 # count shared inodes once toward disk use
            seen.add(key); grand_unique += stt.st_size
        master.append([d.name, str(p.relative_to(BASE_DIR)), stt.st_size])
    grand_files += len(files); grand_listed += nbytes
    print(f"{d.name:40s} {len(files):7d} {human(nbytes):>12s}")
print("-" * 62)
print(f"{'TOTAL (listed)':40s} {grand_files:7d} {human(grand_listed):>12s}")
print(f"{'TOTAL (unique on disk)':40s} {'':>7s} {human(grand_unique):>12s}")
write_manifest("manifest_00_master.csv", master, ["source", "relpath", "size_bytes"])
print("\nMaster manifest -> _manifests/manifest_00_master.csv")
print("Logs            ->", LOG_FILE)
print("\nDone. Re-run any source cell safely — completed files are skipped.")

!python build_tree_view.py .

source                                     files         size
--------------------------------------------------------------
.ipynb_checkpoints                             1       41.9KB
01_figshare_5cities_becsei                    21        1.7GB
02_ena_PRJEB68319_assemblies               38128       69.7GB
03_msystems_resistome                          7      936.0MB
04_ncbi_wastewater_metagenome_527639        5391        6.8GB
06_pubmed39215001_timeseries                   1       754.0B
07_nature_s41467_MAGs                      38399       69.8GB
08_biorxiv_666059_reanalysis                   1       679.0B
--------------------------------------------------------------
TOTAL (listed)                             81949      149.0GB
TOTAL (unique on disk)                                 79.2GB
12:43:35 INFO manifest -> /mmfs2/home/usd.local/john.kangethe/datasets/_manifests/manifest_00_master.csv (81949 rows)

Master manifest -> _manifests/manifest_00_master.csv
Logs            -> /

In [3]:
# ===== CELL A (quota-safe) — UNZIP 04: verify FASTA, then delete each zip =====
import shutil, zipfile, logging, sys
from pathlib import Path
from collections import Counter

BASE_DIR    = Path("~/datasets").expanduser()
NCBI_DIR    = BASE_DIR / "04_ncbi_wastewater_metagenome_527639"
ZIP_DIR     = NCBI_DIR / "genomes"
FASTA_DIR   = NCBI_DIR / "mags_input"          # feed THIS to pathogen_predict.py mags
DELETE_ZIP_AFTER = True                        # frees space as it goes; keeps bad zips
FASTA_DIR.mkdir(parents=True, exist_ok=True)

log = logging.getLogger("unzip"); log.setLevel(logging.INFO)
if not log.handlers:
    h = logging.StreamHandler(sys.stdout)
    h.setFormatter(logging.Formatter("%(asctime)s %(message)s", "%H:%M:%S")); log.addHandler(h)

zips = sorted(ZIP_DIR.glob("*.zip"))
log.info("Unzipping %d archives -> %s (delete_zip_after=%s)", len(zips), FASTA_DIR, DELETE_ZIP_AFTER)

def _extract(zp):
    acc = zp.stem
    dest = FASTA_DIR / f"{acc}.fna"
    if dest.exists() and dest.stat().st_size > 0:           # already done
        if DELETE_ZIP_AFTER and zp.exists():
            try: zp.unlink()
            except OSError: pass
        return "skip"
    try:
        with zipfile.ZipFile(zp) as z:
            fna = [m for m in z.namelist() if m.endswith("_genomic.fna")] or \
                  [m for m in z.namelist() if m.endswith(".fna")]
            if not fna:
                return "no_fna"
            tmp = dest.with_suffix(".fna.part")
            with z.open(fna[0]) as s, open(tmp, "wb") as o:
                shutil.copyfileobj(s, o)
        with open(tmp, "rb") as f:                           # verify it's real FASTA
            head = f.read(1)
        if tmp.stat().st_size == 0 or head != b">":
            tmp.unlink(missing_ok=True); return "bad_fna"
        tmp.replace(dest)
        if DELETE_ZIP_AFTER:
            try: zp.unlink()
            except OSError: pass
        return "ok"
    except zipfile.BadZipFile:
        return "bad_zip"
    except OSError as e:                                     # quota etc. -> stop cleanly
        return f"OSERROR:{e}"
    except Exception as e:
        return f"fail:{e}"

results = []
for i, zp in enumerate(zips, 1):
    r = _extract(zp); results.append(r)
    if r.startswith("OSERROR"):
        log.error("STOPPED at %s -> %s  (free more space, then re-run this cell)", zp.name, r)
        break
    if i % 250 == 0 or i == len(zips):
        log.info("unzip %d/%d", i, len(zips))

print("Summary:", dict(Counter(x.split(':')[0] for x in results)))
print("FASTA files:", len(list(FASTA_DIR.glob('*.fna'))),
      "| remaining zips (problems/undone):", len(list(ZIP_DIR.glob('*.zip'))))


!python build_tree_view.py .

14:34:51 Unzipping 0 archives -> /mmfs2/home/usd.local/john.kangethe/datasets/04_ncbi_wastewater_metagenome_527639/mags_input (delete_zip_after=True)
Summary: {}
FASTA files: 5385 | remaining zips (problems/undone): 0
Scanning /mmfs2/home/usd.local/john.kangethe/datasets ...
Wrote datasets_tree.html: 12 folders, 5,416 files, 27.15 GB on disk (27.15 GB listed incl. hard-links) -> 0.3 MB HTML
